# Smart Employee Expense Anomaly Detection Using Machine Learning

**Google Colab version**

1. Upload this notebook to Google Colab.
2. Click **Runtime → Run all**.
3. When asked, upload `Smart_Employee_Expense_Anomaly_Detection_Jupyter.zip` (the project ZIP). The notebook will extract the dataset automatically.

Each code cell displays its output below it, including the graphs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

In [ ]:
from google.colab import files
import zipfile, io, os

print('Upload the project ZIP: Smart_Employee_Expense_Anomaly_Detection_Jupyter.zip')
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the project ZIP file (.zip).')

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as project_zip:
    project_zip.extractall('/content')

data_path = '/content/Smart_Employee_Expense_Anomaly_Detection/data/employee_expenses.csv'
df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
print("Columns:", list(df.columns))
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
features = ["Department", "Category", "Location", "Amount",
            "Weekend_Days", "Receipt_Count", "Payment_Method"]
categorical = ["Department", "Category", "Location", "Payment_Method"]
numeric = ["Amount", "Weekend_Days", "Receipt_Count"]

X = df[features].copy()
for col in numeric:
    X[col] = pd.to_numeric(X[col], errors="coerce")
X = X.dropna(subset=features)
print("Records Ready for ML:", len(X))

In [ ]:
df["Amount"].plot(kind="hist", bins=12, edgecolor="black", color="white")
plt.xlabel("Expense Amount")
plt.ylabel("Number of Expenses")
plt.title("Employee Expense Amount Distribution")
plt.tight_layout()
plt.show()

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", IsolationForest(n_estimators=300,
                              contamination=0.05,
                              random_state=42))
])
print("Machine Learning Pipeline Created.")

In [ ]:
model.fit(X)
print("Isolation Forest Model Training Completed!")

In [ ]:
predictions = model.predict(X)
results = df.loc[X.index].copy()
results["Status"] = ["Anomaly" if value == -1 else "Normal"
                     for value in predictions]
results[["Employee_ID", "Category", "Amount", "Status"]].head(10)

In [ ]:
normal_count = int((results["Status"] == "Normal").sum())
anomaly_count = int((results["Status"] == "Anomaly").sum())
anomaly_rate = anomaly_count / len(results) * 100

print("SMART EMPLOYEE EXPENSE ANOMALY DETECTION")
print("Model: Isolation Forest")
print("Records Analyzed:", len(results))
print("Normal Expenses:", normal_count)
print("Anomalies Flagged for Review:", anomaly_count)
print("Anomaly Rate: {:.2f}%".format(anomaly_rate))
print("Total Expense Amount: {:.2f}".format(results["Amount"].sum()))

In [ ]:
status_counts = results["Status"].value_counts().reindex(
    ["Normal", "Anomaly"], fill_value=0
)
status_counts.plot(kind="bar", color=["white", "0.65"], edgecolor="black")
plt.xlabel("Expense Status")
plt.ylabel("Number of Records")
plt.title("Normal Expenses vs Anomalies")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
results[results["Status"] == "Anomaly"][
    ["Employee_ID", "Department", "Category", "Amount", "Payment_Method", "Status"]
].head(10)